# Global Wheat Detection — Submission Notebook (Internet: Off)

Notebook này **không chứa logic train/tiền xử lý** — toàn bộ nằm ở `src/` (viết ở VS Code, commit qua Git). Notebook chỉ:
1. Cài offline các gói pip mà Kaggle chưa có sẵn (wheel đóng gói sẵn trong Dataset).
2. Nạp `src/` từ Dataset vào `sys.path`.
3. Load `best.pt` (cũng nằm trong Dataset) và chạy `run_inference()`.

**Trước khi Submit:**
- Add-input: Dataset offline này (đẩy lên bằng `scripts/deploy_kaggle.py`) **và** cuộc thi "Global Wheat Detection" (tab Competitions).
- Notebook Settings -> **Internet: Off** (bắt buộc với Code Competition).
- Sửa `DATASET_SLUG` ở cell dưới nếu bạn đặt `--slug` khác lúc deploy (mặc định đã khớp `wheat-yolov8-offline-bundle`) — cell tự dò đường dẫn mount thật, không cần biết trước `/kaggle/input/...` là gì.

In [ ]:
# Đổi nếu bạn đặt --slug khác lúc chạy scripts/deploy_kaggle.py
DATASET_SLUG = "wheat-yolov8-offline-bundle"

# Tự dò đường dẫn mount thật của dataset — Kaggle có 2 kiểu mount tuỳ phiên bản
# môi trường: "/kaggle/input/datasets/<username>/<slug>/" (mới) hoặc
# "/kaggle/input/<slug>/" (cũ). Dò tự động để khỏi phải sửa tay mỗi lần đổi máy/session.
import glob

_candidates = glob.glob(f"/kaggle/input/datasets/*/{DATASET_SLUG}") + glob.glob(f"/kaggle/input/{DATASET_SLUG}")
if not _candidates:
    raise FileNotFoundError(
        f"Không tìm thấy dataset '{DATASET_SLUG}' trong /kaggle/input/ "
        "-> kiểm tra đã Add Input đúng dataset chưa (chạy `!ls /kaggle/input/datasets/` để xem thực tế)."
    )
BUNDLE_DIR = _candidates[0]
print("BUNDLE_DIR =", BUNDLE_DIR)

# Đường dẫn mount của cuộc thi (add qua tab "Competitions" — nếu add qua Datasets
# thường thì bỏ "competitions/" cho khớp).
COMPETITION_DIR = "/kaggle/input/competitions/global-wheat-detection"

OUTPUT_CSV = "submission.csv"
CONF_THRESHOLD = 0.3
IOU_THRESHOLD = 0.5
IMGSZ = 1024

In [ ]:
# Cài offline từ wheel đã đóng gói sẵn trong Dataset — KHÔNG cần Internet.
# Nếu image Kaggle đã có sẵn đúng bản ultralytics thì lệnh này chỉ mất vài giây
# (pip thấy requirement đã thoả và bỏ qua).
!pip install --no-index --find-links="{BUNDLE_DIR}/packages" -q ultralytics ultralytics-thop py-cpuinfo

import ultralytics
print("ultralytics:", ultralytics.__version__)

In [ ]:
import sys

sys.path.insert(0, f"{BUNDLE_DIR}/source")

from src.infer import run_inference

In [ ]:
submission_path = run_inference(
    weights_path=f"{BUNDLE_DIR}/weights/best.pt",
    source_dir=f"{COMPETITION_DIR}/test",
    output_csv=OUTPUT_CSV,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    imgsz=IMGSZ,
    sample_submission=f"{COMPETITION_DIR}/sample_submission.csv",
)

In [ ]:
# Kiểm tra nhanh trước khi Submit: đúng số dòng / đúng cột / xem thử vài dòng đầu.
import pandas as pd

sub = pd.read_csv(submission_path)
sample = pd.read_csv(f"{COMPETITION_DIR}/sample_submission.csv")

assert list(sub.columns) == ["image_id", "PredictionString"], sub.columns
assert len(sub) == len(sample), f"Số dòng lệch: {len(sub)} (bài nộp) vs {len(sample)} (sample)"
assert set(sub["image_id"]) == set(sample["image_id"]), "Thiếu/thừa image_id so với sample_submission"

n_empty = (sub["PredictionString"].fillna("") == "").sum()
print(f"OK: {len(sub)} dòng, {n_empty} ảnh không có box dự đoán nào.")
sub.head()